# Experiment 16: Using LSTM to Predict Future Weather of a City Using Weather Data from Several Other Cities

**Language:** Python  
**Estimated Time:** 1 hour

## Aim
To use an **LSTM (Long Short-Term Memory)** neural network to predict the future temperature of one target city using historical weather data from several other cities.

## Learning Outcomes
- Understand time-series prediction with LSTM
- Use weather data from multiple cities as input features
- Prepare sequential data for deep learning
- Train and evaluate an LSTM model
- Predict future weather values for a target city

## Suggested Dataset Format
Use a CSV file such as `multi_city_weather.csv` with columns like:

- `date`
- `city_A_temp`
- `city_B_temp`
- `city_C_temp`
- `city_D_temp`
- `target_city_temp`

If you do not have a dataset, this notebook also creates a **sample synthetic dataset** so the experiment can run end-to-end.


In [ ]:
# Install if needed (run once)
# !pip install numpy pandas matplotlib scikit-learn tensorflow


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

np.random.seed(42)


In [ ]:
# Option A: Load your own dataset
# Uncomment and edit the filename if you already have weather data.
# df = pd.read_csv("multi_city_weather.csv", parse_dates=["date"])

# Option B: Create a sample synthetic multi-city weather dataset
days = 500
dates = pd.date_range(start="2022-01-01", periods=days, freq="D")

base = 25 + 8*np.sin(np.linspace(0, 12*np.pi, days))
city_A = base + np.random.normal(0, 1.0, days)
city_B = base + 1.5*np.sin(np.linspace(0, 8*np.pi, days)) + np.random.normal(0, 1.2, days)
city_C = base - 1.0*np.cos(np.linspace(0, 10*np.pi, days)) + np.random.normal(0, 1.1, days)
city_D = base + 0.5*np.sin(np.linspace(0, 15*np.pi, days)) + np.random.normal(0, 0.9, days)

# Target city temperature depends partly on other cities + its own seasonal pattern
target_city_temp = (
    0.30*city_A + 0.25*city_B + 0.20*city_C + 0.15*city_D
    + 2*np.sin(np.linspace(0, 6*np.pi, days))
    + np.random.normal(0, 0.8, days)
)

df = pd.DataFrame({
    "date": dates,
    "city_A_temp": city_A,
    "city_B_temp": city_B,
    "city_C_temp": city_C,
    "city_D_temp": city_D,
    "target_city_temp": target_city_temp
})

df.head()


In [ ]:
print("Dataset shape:", df.shape)
df.info()


In [ ]:
# Plot temperature trends
plt.figure(figsize=(12, 5))
plt.plot(df["date"], df["city_A_temp"], label="City A")
plt.plot(df["date"], df["city_B_temp"], label="City B")
plt.plot(df["date"], df["city_C_temp"], label="City C")
plt.plot(df["date"], df["city_D_temp"], label="City D")
plt.plot(df["date"], df["target_city_temp"], label="Target City", linewidth=2)
plt.title("Weather Data from Multiple Cities")
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.legend()
plt.grid(True)
plt.show()


## Feature Selection

We will use weather values from several cities as input and the **target city temperature** as the value to predict.


In [ ]:
feature_cols = ["city_A_temp", "city_B_temp", "city_C_temp", "city_D_temp", "target_city_temp"]
data = df[feature_cols].values

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

scaled_data[:5]


## Create Sequences for LSTM

LSTM expects 3D input:
- samples
- time steps
- features

We will use the previous **30 days** to predict the next day's target city temperature.


In [ ]:
def create_sequences(dataset, look_back=30, target_index=4):
    X, y = [], []
    for i in range(look_back, len(dataset)):
        X.append(dataset[i-look_back:i, :])   # all features for past look_back days
        y.append(dataset[i, target_index])    # target city's next-day temperature
    return np.array(X), np.array(y)

look_back = 30
X, y = create_sequences(scaled_data, look_back=look_back, target_index=4)

print("X shape:", X.shape)  # (samples, time_steps, features)
print("y shape:", y.shape)


In [ ]:
# Train-test split
train_size = int(len(X) * 0.8)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))


## Build the LSTM Model

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")
model.summary()


In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_split=0.1,
    verbose=1
)


In [ ]:
# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


## Make Predictions and Convert Back to Real Temperatures

In [ ]:
y_pred = model.predict(X_test)

# To inverse transform, rebuild arrays with same number of features
dummy_pred = np.zeros((len(y_pred), len(feature_cols)))
dummy_true = np.zeros((len(y_test), len(feature_cols)))

dummy_pred[:, 4] = y_pred.flatten()
dummy_true[:, 4] = y_test.flatten()

y_pred_actual = scaler.inverse_transform(dummy_pred)[:, 4]
y_test_actual = scaler.inverse_transform(dummy_true)[:, 4]

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae = mean_absolute_error(y_test_actual, y_pred_actual)

print("RMSE:", round(rmse, 3))
print("MAE :", round(mae, 3))


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test_actual, label="Actual Target City Temp")
plt.plot(y_pred_actual, label="Predicted Target City Temp")
plt.title("Actual vs Predicted Weather of Target City")
plt.xlabel("Time Step")
plt.ylabel("Temperature")
plt.legend()
plt.grid(True)
plt.show()


## Predict the Next Day Weather

Use the latest 30-day window to predict the next day's target city temperature.


In [ ]:
last_sequence = scaled_data[-look_back:]
last_sequence = np.expand_dims(last_sequence, axis=0)

next_day_scaled = model.predict(last_sequence)

dummy_next = np.zeros((1, len(feature_cols)))
dummy_next[:, 4] = next_day_scaled.flatten()

next_day_temp = scaler.inverse_transform(dummy_next)[:, 4][0]
print("Predicted next day temperature for target city:", round(next_day_temp, 2))


## Conclusion

An LSTM model was implemented to predict the future weather of a target city using weather data from several other cities.  
The notebook demonstrated:
- sequence creation for time-series forecasting,
- training an LSTM model,
- evaluating prediction accuracy,
- and forecasting the next day's temperature.

You can replace the synthetic dataset with a real multi-city weather dataset for more realistic results.
